In [1]:
# HW 1

In [2]:
import cv2 as cv
import numpy as np

PART2_PATH = "Part2_Pictures"
img = cv.imread(f"./{PART2_PATH}/HW1_IMG_CS898BA.png")

print(img.shape)
# BGR instead of RGB since we're using OpenCV

(1536, 2816, 3)


# Part 2

In [3]:
# Part 2.1 of HW 1
from scipy import stats

BGR = ["Blue", "Green", "Red"]

for i in range(3):
    print(f"Stats for {BGR[i]} channel\n---------------------------------")
    channel = img[:,:,i].flatten()

    minimum = np.min(channel)
    maximum = np.max(channel)
    average = np.mean(channel)
    median = np.median(channel)
    mode = stats.mode(channel, keepdims=True)[0][0]
    skew = stats.skew(channel)
    s_range = np.ptp(channel)

    std = np.std(channel)
    var = np.var(channel)

    print(f"Minimum: {minimum}")
    print(f"Maximum: {maximum}")
    print(f"Median: {median}")
    print(f"Average: {average}")
    print(f"Mode: {mode}")
    print(f"Skew: {skew}")
    print(f"Range: {s_range}")
    print(f"Standard Deviation: {std}")
    print(f"Variance: {var}\n")

Stats for Blue channel
---------------------------------
Minimum: 0
Maximum: 255
Median: 10.0
Average: 21.831687002471
Mode: 4
Skew: 1.6796471408588818
Range: 255
Standard Deviation: 26.229604468480193
Variance: 687.9921505729162

Stats for Green channel
---------------------------------
Minimum: 0
Maximum: 255
Median: 16.0
Average: 24.640292543353457
Mode: 10
Skew: 1.7618090532862751
Range: 255
Standard Deviation: 22.22529703598055
Variance: 493.9638283375659

Stats for Red channel
---------------------------------
Minimum: 0
Maximum: 255
Median: 12.0
Average: 20.60784912109375
Mode: 4
Skew: 2.1054486398247265
Range: 255
Standard Deviation: 22.45564546558139
Variance: 504.2560132758861



In [4]:
# Part 2.2 of HW 1

grayscale = cv.cvtColor(img,cv.COLOR_BGR2GRAY)
cv.imwrite(f'./{PART2_PATH}/grayscale-image.png', grayscale)

_, binary = cv.threshold(grayscale, 127, 255, cv.THRESH_BINARY)
cv.imwrite(f'./{PART2_PATH}/binary_image.png', binary)

hsv = cv.cvtColor(img, cv.COLOR_BGR2HSV)
cv.imwrite(f'./{PART2_PATH}/HSV_image.png', hsv)

cielab = cv.cvtColor(img, cv.COLOR_BGR2LAB)
cv.imwrite(f'./{PART2_PATH}/CIELAB_image.png', cielab)

hls = cv.cvtColor(img, cv.COLOR_BGR2HLS)
cv.imwrite(f'./{PART2_PATH}/HLS_image.png', hls)

True

In [5]:
# Part 2.3, 2.4 of HW 1

h, s, v = cv.split(hsv)

v = cv.equalizeHist(v)
normalized = cv.merge([h,s,v])

normalized = cv.cvtColor(normalized, cv.COLOR_HSV2BGR)
cv.imwrite(f'./{PART2_PATH}/Normalized_hsv_image.png', normalized)

True

In [6]:
# Part 2.6 of HW 1
# followed https://docs.opencv.org/5.0/py_tutorials/py_imgproc/py_geometric_transformations/py_geometric_transformations.html
import random

images = [img, grayscale, binary, hsv, cielab, hls, normalized]
allImages = list(images)
rows, cols, channel = img.shape

for index, image in enumerate(images):
    # rotation
    deg = 360/(1+len(images))
    deg = 360 - deg * (index+1)
    
    rotate = cv.getRotationMatrix2D(((cols-1)/2.0,(rows-1)/2.0),deg,1)
    result_image = cv.warpAffine(image,rotate,(cols,rows))
    path = f"./{PART2_PATH}/affine{index + 1}.png"
    cv.imwrite(path, result_image)
    allImages.append(result_image)

    # translation
    rand = random.randint(-5000, 5000)
    translate = np.float32([[1,0,rand % 214],[0, 1, rand % 491]])
    result_image = cv.warpAffine(image,translate,(cols,rows))
    path = f"./{PART2_PATH}/affine{index + 8}.png"
    cv.imwrite(path, result_image)
    allImages.append(result_image)

In [7]:
len(allImages)

21

In [8]:
# Part 2.8 of HW 1
from pathlib import Path

sigma_levels = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]
Path(f"./{PART2_PATH}/blurred_images").mkdir(parents=True, exist_ok=True)


for index, image in enumerate(allImages):
    for sigma in sigma_levels:
        blurred = cv.GaussianBlur(image, (0,0), sigmaX=sigma)
        path = f"./{PART2_PATH}/blurred_images/blurred{index + 1}_sigma{sigma}.png"
        cv.imwrite(path, blurred)

**2.8 Discussion**

The level of sigma seemed to change how grainy the image felt. The biggest difference when I was looking through the images was with blurred7_sigma0.5.png and blurred7_sigma3.5.png where if you zoom in really close the you're able to kind of see the individual pixels easier on the lower sigma than the higher sigma.

# Part 3

In [13]:
import os
import random

blurred_dir = f"./{PART2_PATH}/blurred_images/"
blurred_pictures = [f for f in os.listdir(blurred_dir) if f.endswith(".png")]
normal_pictures = [f for f in os.listdir(f"./{PART2_PATH}/") if f.endswith(".png")]

blurred_pictures = [f"blurred_images/{picture}" for picture in blurred_pictures]
all_pictures = blurred_pictures + normal_pictures

random.shuffle(all_pictures)
subset = all_pictures[0:42]

In [15]:
len(subset)
subset[2]

'blurred_images/blurred15_sigma3.5.png'

In [19]:
PART3_PATH = "./Part3_Pictures"

prewitt_x = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
prewitt_y = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=np.float32)

for file_name in subset:
    img_path = os.path.join(PART2_PATH, file_name)

    image = cv.imread(img_path)
    gray_img = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    
    # sobel
    sobelx = cv.Sobel(gray_img, cv.CV_64F, 1, 0, ksize=3)
    sobely = cv.Sobel(gray_img, cv.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv.magnitude(sobelx, sobely)
    sobel_img = cv.convertScaleAbs(sobel_mag)

    # laplacian
    lap_img = cv.Laplacian(gray_img, cv.CV_64F, ksize=3)
    lap_img = cv.convertScaleAbs(lap_img)

    # canny
    canny_img = cv.Canny(gray_img, 100, 200)

    # prewitt
    x = cv.filter2D(gray_img, cv.CV_64F, prewitt_x)
    y = cv.filter2D(gray_img, cv.CV_64F, prewitt_y)
    combined = cv.magnitude(x, y)
    prewitt_img = cv.convertScaleAbs(combined)

    cv.imwrite(f"{PART3_PATH}/{file_name}_base.png", image)
    cv.imwrite(f"{PART3_PATH}/{file_name}_sobel.png", sobel_img)
    cv.imwrite(f"{PART3_PATH}/{file_name}_laplacian.png", lap_img)
    cv.imwrite(f"{PART3_PATH}/{file_name}_canny.png", canny_img)
    cv.imwrite(f"{PART3_PATH}/{file_name}_prewitt.png", prewitt_img)